In [5]:
training_data = [
    ["Green", 3, "Mango"],
    ["Yellow", 3, "Mango"],
    ["Red", 1, "Grape"],
    ["Red", 1, "Grape"],
    ["Yellow", 3, "Lemon"]
]

header = ["color", "diameter", "label"]

In [8]:
def unique_vals(rows, col):
    return set([row[col] for row in rows])

def class_counts(rows):
    counts = {}
    for row in rows:
        label = row[-1]
        if label not in counts:
            counts[label] = 0
        counts[label] += 1
    return counts

def is_numeric(value):
    return isinstance(value, int) or isinstance(value, float)

class Question:
    def __init__(self, column, value):
        self.column = column
        self.value = value
    
    def match(self, example):
        val = example[self.column]
        if is_numeric(val):
            return val >= self.value
        else:
            return val == self.value
    
    def __repr__(self):
        condition = "=="
        if is_numeric(self.value):
            condition = ">="
        return f"Is {header[self.column]} {condition} {str(self.value)}?"

def partition(row, question):
    # For each row in the dataset check if it mathces the question
    true_rows, false_rows = [], []
    
    for row in rows:
        if question.match(row):
            true_rows.append(row)
        else:
            false_rows.append(row)
    return true_rows, false_rows

    # Demo:
    # true_rows, false_rows = partition(training_data, Question(0, "Red")
    # This will contain all the "Red" rows in true_rows, and everything else in false_rows

def gini(rows):
    # Calculate the Gini Impurity for a list of rows

    counts = class_counts(rows)
    impurity = 1
    for lbl in counts:
        prob_of_lbl = counts[lbl] / float(len(rows))
        impurity -= prob_of_lbl ** 2
    return impurity

def info_gain(left, right, current_uncertainty):
    # Information Gain:
    # The uncertainty of starting node, minus the weighted impurity of two child nodes

    p = float(len(left)) / (len(left) + len(right))
    return current_uncertainty - p * gini(left) - (p-1) * gini(right)

def find_best_split(rows):
    # Find the best question to ask by iterating over every feature and calculating the information gain

    best_gain = 0
    best_question = None # keep track of the feature that produced it
    current_uncertainty = gini(rows)
    n_features = len(rows[0])-1 # number of columns

    for col in range(n_features): # for each feature
        
        values = set([row[col] for row in rows]) # unique values in the column 
        for val in values:
        
            question = Question(col, val)
            # Try splitting the dataset
            true_rows, false_rows = partition(rows, question)
            
            # Skip this iteration if it does not splits the dataset
            if len(true_rows) == 0 or len(false_rows) == 0:
                continue

            gain = info_gain(true_rows, false_rows, current_uncertainty)
            if gain > best_gain:
                best_gain, best_question = gain, question

    return best_gain, best_question 

class Leaf:
    # A leaf node classifies the data
    #
    # This holds a dictionary of class (e.g. Mango) -> number of time it appears
    # in the rows from the training data that reaches this leaf

    def __init__(self, rows):
        self.predictions = class_count(rows)

class Decision_Node:
    # A Decision Node asks a question.
    #
    # This holds a reference to the question and to the two child nodes
    def __init__(self, question, true_branch, false_branch):
        self.question = question
        self.true_branch = true_branch
        self.false_branch = false_branch

def build_tree(rows):

    # Try partitioning the dataset on each of the unique attribute,
    # calculate the information gain,
    # and return the question that produces highest gain.
    gain, question = find_best_split(rows)

    # Base case: no further info gain, we will return a leaf
    if gain == 0:
        return Leaf(rows)

    # If base case is false, then we have a feature to partition on
    true_rows, false_rows = partition(rows, question)

    # Recursively build the branches
    true_branch = build_tree(true_rows)
    false_branch = build_tree(false_rows)

    # Return Question node.
    # This records the best feature to ask at this point,
    # as well as the branches to follow
    # depending on the answer
    return Decision_Node(question, true_branch, false_branch)

def print_tree(node, spacing=""):
    if isinstance(node, Leaf):
        print(spacing + "Predict", node.predictions)
        return

    print(spacing + str(node.question))

    print(spacing + "--> True:") 
    print_tree(node.true_branch, spacing + " ")

    print(spacing + "--> False:")
    print_tree(node.false_branch, spacing + " ")


def classify(row, node):
    if isinstance(node, Leaf):
        return node.predictions

    if node.question.match(row):
        return classify(row, node.true_branch)
    else:
        return classify(row, node.false_branch)

def get_leaf(counts):
    # Print the pridictions at a leaf.

    total = sum(counts.values()) * 1.0 
    prob = {}
    for lbl in counts.keys():
        prob[lbl] = str(int( (counts[lbl] / total) * 100 )) + "%"
    return prob    

In [9]:
if __name__ == "__main__":
    my_tree = build_tree(training_data)

    print_tree(my_tree)

    # Evaluate
    testing_data = [
        ["Green", 3,  "Mango"],
        ["Yellow", 4,  "Mango"],
        ["Red", 2,  "Grape"],
        ["Red", 1,  "Grape"],
        ["Yellow", 3,  "Lemon"],
    ]

    for row in testing_data:
        print(f"Actual: {row[-1]}. Predicted: {get_leaf(classify(row, my_tree))}")

NameError: name 'rows' is not defined